# Лабораторная работа №7
## Деревья
## Ревенко Данила, ФИТ-231


In [ ]:
# Из лекции: стек и базовый класс бинарного дерева
class Stack:
    def __init__(self):
        self.items = []

    def isEmpty(self):
        return self.items == []

    def push(self, item):
        self.items.append(item)

    def pop(self):
        return self.items.pop()

    def peek(self):
        return self.items[-1]

    def size(self):
        return len(self.items)


class BinaryTree:
    def __init__(self, rootObj):
        self.key = rootObj
        self.leftChild = None
        self.rightChild = None

    def insertLeft(self, newNode):
        if self.leftChild is None:
            self.leftChild = BinaryTree(newNode)
        else:
            t = BinaryTree(newNode)
            t.leftChild = self.leftChild
            self.leftChild = t

    def insertRight(self, newNode):
        if self.rightChild is None:
            self.rightChild = BinaryTree(newNode)
        else:
            t = BinaryTree(newNode)
            t.rightChild = self.rightChild
            self.rightChild = t

    def getRightChild(self):
        return self.rightChild

    def getLeftChild(self):
        return self.leftChild

    def setRootVal(self, obj):
        self.key = obj

    def getRootVal(self):
        return self.key

### № 1
a) Перепишите функции buildParseTree и evaluate из лекционных материалов так,
чтобы они работали с логическими выражениями, то есть с операторами «и», «или»,
«не» и операндами «истина» и «ложь». Помните, что «не» – унарный оператор.

b) Реализуйте функцию printexp, которая будет принимать дерево синтаксического
разбора и возвращать строку соответствующего ему логического выражения.

In [ ]:
# Функции для логических выражений: построение дерева разбора, вычисление и восстановление выражения
def buildParseTreeBool(expr):
    tokens = expr.split()

    def parse(tokens, index=0):
        token = tokens[index]

        if token in ("истина", "ложь"):
            node = BinaryTree(token == "истина")
            return node, index + 1

        if token == "(":
            index += 1
            if tokens[index] == "не":
                index += 1
                subtree, index = parse(tokens, index)
                node = BinaryTree("не")
                node.rightChild = subtree
                if tokens[index] != ")":
                    raise ValueError("Ожидалась ) после 'не'")
                return node, index + 1
            else:
                left, index = parse(tokens, index)
                op = tokens[index]
                if op not in ("и", "или"):
                    raise ValueError("Ожидался 'и' или 'или'")
                index += 1
                right, index = parse(tokens, index)
                if tokens[index] != ")":
                    raise ValueError("Ожидалась ) после бинарного выражения")
                node = BinaryTree(op)
                node.leftChild = left
                node.rightChild = right
                return node, index + 1

        raise ValueError(f"Неожиданный токен: {token}")

    tree, idx = parse(tokens, 0)
    if idx != len(tokens):
        raise ValueError("Лишние токены в конце выражения")
    return tree


def evaluateBool(parseTree):
    """Вычисление логического выражения по дереву разбора."""
    root = parseTree.getRootVal()
    left = parseTree.getLeftChild()
    right = parseTree.getRightChild()

    if root == "и":
        return evaluateBool(left) and evaluateBool(right)
    if root == "или":
        return evaluateBool(left) or evaluateBool(right)
    if root == "не":
        return not evaluateBool(right)
    return root  # лист: True / False


def printexp(tree):
    """Восстановление строкового логического выражения из дерева разбора."""
    root = tree.getRootVal()
    left = tree.getLeftChild()
    right = tree.getRightChild()

    if root in ("и", "или"):
        return f"( {printexp(left)} {root} {printexp(right)} )"
    if root == "не":
        return f"( не {printexp(right)} )"
    return "истина" if root is True else "ложь"


# Тесты для логических выражений
expr1 = "( истина и ( не ложь ) )"
pt1 = buildParseTreeBool(expr1)
print(printexp(pt1))        # Ожидается: ( истина и ( не ложь ) )
print(evaluateBool(pt1))    # Ожидается: True

expr2 = "( ( истина или ложь ) и ( не истина ) )"
pt2 = buildParseTreeBool(expr2)
print(printexp(pt2))        # Ожидается: ( ( истина или ложь ) и ( не истина ) )
print(evaluateBool(pt2))    # Ожидается: False

expr3 = "( ( истина и ложь ) или ( не ( истина и ложь ) ) )"
pt3 = buildParseTreeBool(expr3)
print(printexp(pt3))        # Ожидается: ( ( истина и ложь ) или ( не ( истина и ложь ) ) )
print(evaluateBool(pt3))    # Ожидается: True

( истина и ( не ложь ) )
True
( ( истина или ложь ) и ( не истина ) )
False
( ( истина и ложь ) или ( не ( истина и ложь ) ) )
True


### №2
a) Реализуйте функцию has_no_duplicates, которая принимает в качестве аргумента
двоичное дерево поиска и возвращает «истину» если в нем нет равных ключей и
«ложь» в противном случае.

b) Измените реализацию двоичного дерева поиска из лекционных материалов так,
чтобы оно правильно работало с дубликатами ключей. Т.е., если ключ в дереве уже
присутствует, то новое значение должно заменить старое, вместо того, чтобы
добавлять новый узел с тем же ключом. Проверьте правильность вашего решения
функцией has_no_duplicates.


In [ ]:
# Из лекции: реализация двоичного дерева поиска (BST)
class TreeNode:
    def __init__(self, key, val, left=None, right=None, parent=None):
        self.key = key
        self.payload = val
        self.leftChild = left
        self.rightChild = right
        self.parent = parent

    def hasLeftChild(self):
        return self.leftChild

    def hasRightChild(self):
        return self.rightChild

    def isLeftChild(self):
        return self.parent and self.parent.leftChild == self

    def isRightChild(self):
        return self.parent and self.parent.rightChild == self

    def isRoot(self):
        return not self.parent

    def isLeaf(self):
        return not (self.rightChild or self.leftChild)

    def hasAnyChildren(self):
        return self.rightChild or self.leftChild

    def hasBothChildren(self):
        return self.rightChild and self.leftChild

    def replaceNodeData(self, key, value, lc, rc):
        self.key = key
        self.payload = value
        self.leftChild = lc
        self.rightChild = rc
        if self.hasLeftChild():
            self.leftChild.parent = self
        if self.hasRightChild():
            self.rightChild.parent = self


class BinarySearchTree:

    def __init__(self):
        self.root = None
        self.size = 0

    def length(self):
        return self.size

    def __len__(self):
        return self.size

    def put(self, key, val):
        if self.root:
            self._put(key, val, self.root)
        else:
            self.root = TreeNode(key, val)
        self.size = self.size + 1

    def _put(self, key, val, currentNode):
        if key < currentNode.key:
            if currentNode.hasLeftChild():
                self._put(key, val, currentNode.leftChild)
            else:
                currentNode.leftChild = TreeNode(key, val, parent=currentNode)
        else:
            if currentNode.hasRightChild():
                self._put(key, val, currentNode.rightChild)
            else:
                currentNode.rightChild = TreeNode(key, val, parent=currentNode)

    def __setitem__(self, k, v):
        self.put(k, v)

    def get(self, key):
        if self.root:
            res = self._get(key, self.root)
            if res:
                return res.payload
            else:
                return None
        else:
            return None

    def _get(self, key, currentNode):
        if not currentNode:
            return None
        elif currentNode.key == key:
            return currentNode
        elif key < currentNode.key:
            return self._get(key, currentNode.leftChild)
        else:
            return self._get(key, currentNode.rightChild)

    def __getitem__(self, key):
        return self.get(key)

    def __contains__(self, key):
        if self._get(key, self.root):
            return True
        else:
            return False

    def delete(self, key):
        if self.size > 1:
            nodeToRemove = self._get(key, self.root)
            if nodeToRemove:
                self.remove(nodeToRemove)
                self.size = self.size - 1
            else:
                raise KeyError('Error, key not in tree')
        elif self.size == 1 and self.root.key == key:
            self.root = None
            self.size = self.size - 1
        else:
            raise KeyError('Error, key not in tree')

    def __delitem__(self, key):
        self.delete(key)

    def spliceOut(self):
        if self.isLeaf():
            if self.isLeftChild():
                self.parent.leftChild = None
            else:
                self.parent.rightChild = None
        elif self.hasAnyChildren():
            if self.hasLeftChild():
                if self.isLeftChild():
                    self.parent.leftChild = self.leftChild
                else:
                    self.parent.rightChild = self.leftChild
                self.leftChild.parent = self.parent
            else:
                if self.isLeftChild():
                    self.parent.leftChild = self.rightChild
                else:
                    self.parent.rightChild = self.rightChild
                self.rightChild.parent = self.parent

    def findSuccessor(self):
        succ = None
        if self.hasRightChild():
            succ = self.rightChild.findMin()
        else:
            if self.parent:
                if self.isLeftChild():
                    succ = self.parent
                else:
                    self.parent.rightChild = None
                    succ = self.parent.findSuccessor()
                    self.parent.rightChild = self
        return succ

    def findMin(self):
        current = self
        while current.hasLeftChild():
            current = current.leftChild
        return current

    def remove(self, currentNode):
        if currentNode.isLeaf():  # leaf
            if currentNode == currentNode.parent.leftChild:
                currentNode.parent.leftChild = None
            else:
                currentNode.parent.rightChild = None
        elif currentNode.hasBothChildren():  # interior
            succ = currentNode.findSuccessor()
            succ.spliceOut()
            currentNode.key = succ.key
            currentNode.payload = succ.payload

        else:  # this node has one child
            if currentNode.hasLeftChild():
                if currentNode.isLeftChild():
                    currentNode.leftChild.parent = currentNode.parent
                    currentNode.parent.leftChild = currentNode.leftChild
                elif currentNode.isRightChild():
                    currentNode.leftChild.parent = currentNode.parent
                    currentNode.parent.rightChild = currentNode.leftChild
                else:
                    currentNode.replaceNodeData(currentNode.leftChild.key,
                                                currentNode.leftChild.payload,
                                                currentNode.leftChild.leftChild,
                                                currentNode.leftChild.rightChild)
            else:
                if currentNode.isLeftChild():
                    currentNode.rightChild.parent = currentNode.parent
                    currentNode.parent.leftChild = currentNode.rightChild
                elif currentNode.isRightChild():
                    currentNode.rightChild.parent = currentNode.parent
                    currentNode.parent.rightChild = currentNode.rightChild
                else:
                    currentNode.replaceNodeData(currentNode.rightChild.key,
                                                currentNode.rightChild.payload,
                                                currentNode.rightChild.leftChild,
                                                currentNode.rightChild.rightChild)

In [ ]:
# Функция для проверки отсутствия дубликатов ключей в BST
def has_no_duplicates(bst: BinarySearchTree):
    """Возвращает True, если в двоичном дереве поиска нет дублирующихся ключей."""
    seen = set()

    def traverse(node: TreeNode):
        if node is None:
            return True
        if node.key in seen:
            return False
        seen.add(node.key)
        return traverse(node.leftChild) and traverse(node.rightChild)

    return traverse(bst.root)


# Тесты для проверки дубликатов в BST (исходная реализация дерева)
t = BinarySearchTree()
t[3] = "red"
t[3] = "blue"
t[5] = "red"
print(has_no_duplicates(t))  # Ожидается: False (узлы с ключом 3 дублируются)


# Изменённая реализация _put: при одинаковом ключе обновляет значение в узле
def _put(self, key, val, currentNode):
    if key < currentNode.key:
        if currentNode.hasLeftChild():
            self._put(key, val, currentNode.leftChild)
        else:
            currentNode.leftChild = TreeNode(key, val, parent=currentNode)
    elif key > currentNode.key:
        if currentNode.hasRightChild():
            self._put(key, val, currentNode.rightChild)
        else:
            currentNode.rightChild = TreeNode(key, val, parent=currentNode)
    else:
        # key == currentNode.key: обновляем значение payload вместо добавления нового узла
        currentNode.payload = val

# Привязываем новую реализацию _put к уже существующему классу
BinarySearchTree._put = _put


# Тесты для корректной работы с дубликатами (новая реализация)
t1 = BinarySearchTree()
t1[3] = "red"
t1[1] = "blue"
t1[5] = "green"
print(has_no_duplicates(t1))  # Ожидается: True (все ключи уникальны)

t2 = BinarySearchTree()
t2[3] = "first"
t2[3] = "second"
t2[3] = "final"
print(t2[3])                  # Ожидается: final (значение для ключа 3 обновляется)
print(has_no_duplicates(t2))  # Ожидается: True (дубликатов ключей нет)

False
True
final
True


### №3
Суть игры «Животные» заключается в следующем: программа пытается угадать
задуманное пользователем животное и при этом самообучается, то есть постепенно
совершенствуется. Вся необходимая информация хранится в бинарном дереве, каждая
вершина которого содержит вопрос, предполагающий ответ «да» или «нет».
Последовательный выбор одного из этих ответов ведет программу вниз по
соответствующим ветвям до терминальной вершины, где и находится название животного.
Если программа совершает ошибку, она просит пользователя ввести уточняющий вопрос,
который позволил бы ей прийти к верному решению, затем добавляет его в новую
внутреннюю вершину и создает для нее свои листья.

Предположим, текущая база знаний программы выглядит так, как на рисунке, и
пользователь загадал змею. В таблице приводятся список вопросов, которые задает
программа, и полученные ответы пользователя.

Если задуман жираф, вопросы и ответы прозвучат иначе, а программа добавит в дерево
уточняющую информацию и нужное животное.

Реализуйте эту игру, используя бинарное дерево для хранения вопросов и ответов.


In [ ]:
# Вспомогательная функция для игры «Животные»: проверка, что узел — лист (животное)
def is_leaf(node: BinaryTree):
    return node.getLeftChild() is None and node.getRightChild() is None


# Функция для одной игровой сессии «Животные» от корня дерева
def play_animals_game(root: BinaryTree):
    current = root

    # Спускаемся по дереву по ответам пользователя, пока не дойдём до животного
    while not is_leaf(current):
        question = current.getRootVal()
        answer = input(question + " (да/нет): ").strip().lower()
        if answer.startswith("д"):  # ответ «да» — идём вправо
            child = current.getRightChild()
        else:                        # ответ «нет» — идём влево
            child = current.getLeftChild()

        if child is None:
            print("Не знаю, что дальше спросить.")
            return

        current = child

    # Дошли до листа с предполагаемым животным
    animal = current.getRootVal()
    answer = input(f"Вы загадали {animal}? (да/нет): ").strip().lower()
    if answer.startswith("д"):
        print("Ура! Я угадал.")
        return

    # Программа ошиблась — просим пользователя обучить её
    user_animal = input("Кого вы загадали? ").strip()
    new_question = input(
        f"Введите вопрос, который отличает {user_animal} от {animal}.\n"
        f"На этот вопрос для {user_animal} ответ должен быть 'да': "
    ).strip()

    old_animal = animal
    current.setRootVal(new_question)
    current.leftChild = BinaryTree(old_animal)     # ответ «нет»
    current.rightChild = BinaryTree(user_animal)   # ответ «да»

    print("Спасибо! Я стал умнее.")


# Инициализация начального дерева вопросов и животных для игры
root = BinaryTree("Это млекопитающее?")

# Левая ветка (ответ «нет» на первый вопрос)
not_mammal = BinaryTree("Оно покрыто чешуей?")
root.leftChild = not_mammal

# Правая ветка (ответ «да» на первый вопрос)
mammal = BinaryTree("Оно лает?")
root.rightChild = mammal

# Поддерево для млекопитающих
mammal.leftChild = BinaryTree("Кошка")   # ответ «нет»
mammal.rightChild = BinaryTree("Собака") # ответ «да»

# Поддерево для немлекопитающих
not_mammal.rightChild = BinaryTree("Оно дышит в воде?")  # ответ «да»
not_mammal.leftChild = BinaryTree("Птица")               # ответ «нет»

water_question = not_mammal.getRightChild()
water_question.leftChild = BinaryTree("Змея")  # ответ «нет»
water_question.rightChild = BinaryTree("Рыба") # ответ «да»


# Цикл тестового запуска игры
while True:
    play_animals_game(root)
    again = input("Сыграем ещё раз? (да/нет): ").strip().lower()
    if not again.startswith("д"):
        break

Спасибо! Я стал умнее.


### №4
Создайте двоичную кучу с ограниченным размером. Другими словами, куча может
отслеживать только n важных элементов. Если её размер становится больше, то наименее
приоритетный элемент отбрасывается.

In [ ]:
# Из лекции: реализация двоичной min-heap
class BinHeap:
    def __init__(self):
        self.heapList = [0]
        self.currentSize = 0

    def percUp(self, i):
        while i // 2 > 0:
            if self.heapList[i] < self.heapList[i // 2]:
                tmp = self.heapList[i // 2]
                self.heapList[i // 2] = self.heapList[i]
                self.heapList[i] = tmp
            i = i // 2

    def insert(self, k):
        self.heapList.append(k)
        self.currentSize = self.currentSize + 1
        self.percUp(self.currentSize)

    def percDown(self, i):
        while (i * 2) <= self.currentSize:
            mc = self.minChild(i)
            if self.heapList[i] > self.heapList[mc]:
                tmp = self.heapList[i]
                self.heapList[i] = self.heapList[mc]
                self.heapList[mc] = tmp
            i = mc

    def minChild(self, i):
        if i * 2 + 1 > self.currentSize:
            return i * 2
        else:
            if self.heapList[i * 2] < self.heapList[i * 2 + 1]:
                return i * 2
            else:
                return i * 2 + 1

    def delMin(self):
        retval = self.heapList[1]
        self.heapList[1] = self.heapList[self.currentSize]
        self.currentSize = self.currentSize - 1
        self.heapList.pop()
        self.percDown(1)
        return retval

    def buildHeap(self, alist):
        i = len(alist) // 2
        self.currentSize = len(alist)
        self.heapList = [0] + alist[:]
        while (i > 0):
            self.percDown(i)
            i = i - 1

In [ ]:
# Ограниченная по размеру двоичная куча (min-heap, которая хранит только N наибольших элементов)
class BoundedBinHeap(BinHeap):
    def __init__(self, capacity):
        super().__init__()
        self.capacity = capacity

    def insert(self, k):
        """Вставка элемента с учётом ограничения по размеру.
        Если куча заполнена и новый элемент больше текущего минимума,
        он добавляется, а наименьший элемент удаляется."""
        # Если в куче ещё меньше capacity элементов — просто вставляем
        if self.currentSize < self.capacity:
            super().insert(k)
        else:
            # Если новый элемент важнее (больше) минимального — вставляем его,
            # а затем выбрасываем минимум (наименее приоритетный)
            if k > self.heapList[1]:
                super().insert(k)
                super().delMin()
            # Иначе ничего не делаем — элемент менее приоритетен, чем все в куче

    def items(self):
        """Возврат текущих элементов кучи (порядок произвольный)."""
        return self.heapList[1:]


# Тесты для ограниченной по размеру двоичной кучи
h = BoundedBinHeap(3)
for x in [5, 1, 10, 3, 7, 2]:
    h.insert(x)

print(h.items())       # Ожидается: три наибольших числа, например [5, 10, 7] (порядок не важен)
print(h.delMin())      # Ожидается: наименьший из оставшихся трёх
print(h.items())       # Ожидаются два наибольших из исходного набора

[5, 7, 10]
5
[7, 10]


### №5
Реализуйте двоичную кучу как max heap на основание варианта min heap из лекционных
материалов.

In [ ]:
# Реализация двоичной кучи как max-heap
class MaxBinHeap:
    def __init__(self):
        self.heapList = [0]
        self.currentSize = 0

    def percUp(self, i):
        while i // 2 > 0:
            if self.heapList[i] > self.heapList[i // 2]:
                tmp = self.heapList[i // 2]
                self.heapList[i // 2] = self.heapList[i]
                self.heapList[i] = tmp
            i = i // 2

    def insert(self, k):
        self.heapList.append(k)
        self.currentSize += 1
        self.percUp(self.currentSize)

    def percDown(self, i):
        while i * 2 <= self.currentSize:
            mc = self.maxChild(i)
            if self.heapList[i] < self.heapList[mc]:
                tmp = self.heapList[i]
                self.heapList[i] = self.heapList[mc]
                self.heapList[mc] = tmp
            i = mc

    def maxChild(self, i):
        if i * 2 + 1 > self.currentSize:
            return i * 2
        else:
            if self.heapList[i * 2] > self.heapList[i * 2 + 1]:
                return i * 2
            else:
                return i * 2 + 1

    def delMax(self):
        """Удаление и возврат максимального элемента из кучи."""
        retval = self.heapList[1]
        self.heapList[1] = self.heapList[self.currentSize]
        self.currentSize -= 1
        self.heapList.pop()
        if self.currentSize > 0:
            self.percDown(1)
        return retval

    def buildHeap(self, alist):
        """Построение max-heap из произвольного списка."""
        i = len(alist) // 2
        self.currentSize = len(alist)
        self.heapList = [0] + alist[:]
        while i > 0:
            self.percDown(i)
            i -= 1


# Тесты для max-heap
mh = MaxBinHeap()
mh.buildHeap([9, 5, 6, 2, 3])

print(mh.delMax())  # Ожидается: 9
print(mh.delMax())  # Ожидается: 6
print(mh.delMax())  # Ожидается: 5
print(mh.delMax())  # Ожидается: 3
print(mh.delMax())  # Ожидается: 2

9
6
5
3
2


### №6
Используя класс BinaryHeap из лекционных материалов, реализуйте новый класс очереди с
приоритетом PriorityQueue. Он должен содержать конструктор и методы enqueue и dequeue.

In [ ]:
# Очередь с приоритетом на основе двоичной min-heap
class PriorityQueue:
    # min-priority очередь: меньший ключ = более высокий приоритет
    def __init__(self):
        self._heap = BinHeap()

    def enqueue(self, item):
        """Добавить элемент в очередь с приоритетом."""
        self._heap.insert(item)

    def dequeue(self):
        """Извлечь элемент с наивысшим приоритетом (минимальное значение)."""
        return self._heap.delMin()


# Тесты для очереди с приоритетом
pq = PriorityQueue()
pq.enqueue(5)
pq.enqueue(1)
pq.enqueue(3)

print(pq.dequeue())  # Ожидается: 1
print(pq.dequeue())  # Ожидается: 3
print(pq.dequeue())  # Ожидается: 5

1
3
5
